# ใบงานนักเรียน: สร้าง RAG Chatbot ด้วย BGE-M3 และ Ollama 🤖

ชื่อ ____________________ ชั้น ______ เลขที่ ______

กติกา:

- อ่านโจทย์แล้วพิมพ์โค้ดในเซลล์ `TODO`
- รันเซลล์จากบนลงล่าง
- ตรวจผลในแต่ละ Checkpoint ก่อนทำขั้นต่อไป
- หากติดขัด ให้ดูคำใบ้ก่อนเปิดไฟล์ Solution

## เป้าหมายการเรียนรู้

เมื่อจบบทเรียน นักเรียนจะสามารถ:

1. อธิบายขั้นตอนของ RAG ได้
2. ทำความสะอาดและแบ่งเอกสารเป็น chunks
3. ใช้ BGE-M3 เปลี่ยนข้อความเป็น embedding
4. ใช้ cosine similarity ค้นหาข้อมูลที่เกี่ยวข้อง
5. ส่ง Context และคำถามให้ Qwen ผ่าน OpenAI-compatible API
6. สร้าง RAG Chatbot ที่จำบทสนทนาได้

## 1) รู้จัก RAG

RAG ย่อมาจาก **Retrieval-Augmented Generation** เป็นการค้นข้อมูลที่เกี่ยวข้องก่อน แล้วส่งข้อมูลนั้นให้โมเดลช่วยสร้างคำตอบ

```text
เอกสาร → ทำความสะอาด → แบ่ง Chunk → Embedding → เก็บ Vector
คำถาม → Embedding → ค้นหา Chunk → สร้าง Context → Qwen → คำตอบ
```

คำถาม: เพราะเหตุใดเราจึงไม่ส่งเอกสารทั้งหมดให้โมเดลทุกครั้ง?

## 2) เตรียมโปรแกรมและโมเดล

ก่อนเริ่ม ให้เปิด Ollama และติดตั้งโมเดลใน Terminal:

```bash
ollama pull bge-m3
ollama pull qwen3.5:4b
```

จากนั้นติดตั้ง Python packages ที่จำเป็นในเซลล์ด้านล่าง

In [2]:
# TODO: ติดตั้ง openai, requests, numpy และ scikit-learn ด้วย %pip install

from openai import OpenAI
OLLAMA_BASE_URL = "http://localhost:11434/v1/"
client = OpenAI(
    base_url=OLLAMA_BASE_URL,
    api_key="ollama",  # Ollama ต้องการค่า แต่จะไม่ตรวจสอบ key นี้
)

## 3) ตรวจการเชื่อมต่อ Ollama

ใช้ `requests.get()` เรียก `http://localhost:11434/api/tags` แล้วแสดงชื่อโมเดลที่พบ

คำใบ้: เรียก `r.raise_for_status()` ก่อนอ่าน `r.json()`

In [ ]:
# TODO: import requests และตรวจว่า Ollama เชื่อมต่อได้

OLLAMA_HOST = "http://localhost:11434"

### Checkpoint 1

- [ ] เชื่อมต่อ Ollama สำเร็จ
- [ ] พบโมเดล `bge-m3`
- [ ] พบโมเดล `qwen3.5:4b`

## 4) สร้างเอกสาร

สร้าง list ชื่อ `documents` ที่มีข้อความอย่างน้อย 3 ประโยคเกี่ยวกับประเทศไทย แล้วแสดงจำนวนเอกสาร

In [6]:
# TODO: สร้าง documents อย่างน้อย 3 ประโยค
documents = [
    "เควินชอบเล่น roblox ",
    "น้องพีท ชอบกิน pizza ",
    "poom เก่งคณิต",
    "note นั่งสมาธิเก่ง",
    "    "
]


## 5) ทำความสะอาดข้อความ

สร้าง `clean_docs` โดย:

1. วนอ่านข้อความแต่ละรายการ
2. ใช้ `.strip()` ลบช่องว่างหัวท้าย
3. เก็บเฉพาะข้อความที่ไม่ว่าง

In [8]:
# TODO: ทำความสะอาด documents
clean_docs = []
print("Before cleaning Document : ", len(documents))
for document in documents:
    document = document.strip()
    
    if document:
        clean_docs.append(document)
print( "Document : ",len(clean_docs))

Before cleaning Document :  5
Document :  4


## 6) แบ่งข้อความเป็น Chunk

แบ่งเอกสารแต่ละรายการเป็นส่วนละ 100 ตัวอักษร แล้วเก็บใน `chunks`

คำใบ้:

```python
for i in range(0, len(document), 100):
    ...
```

In [10]:
# TODO: สร้าง chunks จาก clean_docs
chunks = []

for document in clean_docs:
    for i in range(0,len(document),100):
        chunk = document[i:i + 100]
        chunks.append(chunk)
print("Chunks : ", len(chunks))

Chunks :  4


### Checkpoint 2

- [ ] `clean_docs` ไม่มีข้อความว่าง
- [ ] `chunks` มีอย่างน้อย 3 รายการ
- [ ] แต่ละ chunk ยาวไม่เกิน 100 ตัวอักษร

## 7) สร้าง Embedding ด้วย BGE-M3

เขียนฟังก์ชัน `embed(text)` เพื่อส่งข้อความไปยัง `/api/embed`

ข้อมูล JSON ที่ต้องส่ง:

```python
{"model": "bge-m3", "input": text}
```

ฟังก์ชันต้องคืนค่า `r.json()["embeddings"][0]`

In [ ]:
# TODO: เขียนฟังก์ชัน embed(text)
def embed(text):
    pass

## 8) สร้าง Vector Store

เรียก `embed()` กับทุก chunk แล้วเก็บผลใน `embeddings` จากนั้นจับคู่ `chunks` และ `embeddings` ด้วย `zip()`

In [ ]:
# TODO: สร้าง embeddings และ vector_store

## 9) รับคำถามและค้นหา Top 3

กำหนดคำถามใน `query` แล้วทำตามลำดับ:

1. สร้าง `query_vec`
2. คำนวณ `cosine_similarity` ระหว่างคำถามกับ embeddings
3. เรียงคะแนนจากมากไปน้อย
4. เก็บ index 3 อันดับแรกใน `top`
5. แสดงคะแนนและ chunk

In [ ]:
# TODO: import numpy และ cosine_similarity แล้วค้นหา Top 3
query = "เมืองหลวงของประเทศไทยคืออะไร"

### Checkpoint 3

ดูผลการค้นหาแล้วตอบ:

1. Chunk อันดับแรกเกี่ยวข้องกับคำถามหรือไม่?
2. คะแนนของอันดับแรกเป็นเท่าใด?
3. เพราะเหตุใดคะแนนมากจึงควรอยู่ก่อนคะแนนน้อย?

## 10) สร้าง Context

นำ chunks จาก index ใน `top` มาต่อกันด้วย `\n` แล้วสร้าง prompt รูปแบบนี้:

```text
Context:
ข้อมูลที่ค้นพบ

Question: คำถามของผู้ใช้
```

In [ ]:
# TODO: สร้าง context และ prompt แล้ว print ดูผล

## 11) ส่ง Prompt ให้ Qwen

ใช้ OpenAI Python SDK เชื่อมกับ Ollama แบบ OpenAI-compatible:

- `base_url` คือ `http://localhost:11434/v1/`
- `api_key` ใช้ข้อความ `ollama`
- `MODEL` คือ `qwen3.5:4b`

กำหนด system message ให้ตอบจาก Context เท่านั้น และบอกว่าไม่พบข้อมูลเมื่อ Context ไม่มีคำตอบ

In [ ]:
# TODO: import OpenAI, สร้าง client และส่ง prompt ให้ qwen3.5:4b
# TODO: อ่าน response.choices[0].message.content แล้วแสดงคำตอบ

### Checkpoint 4

- [ ] Qwen ตอบคำถามเป็นภาษาไทย
- [ ] คำตอบสอดคล้องกับ Context
- [ ] โค้ดใช้ OpenAI SDK แต่โมเดลยังทำงานใน Ollama

คำถาม: เหตุใด `api_key="ollama"` จึงไม่ใช่ OpenAI API key จริง?

# 12) Mini Project: RAG Chatbot ที่มีความจำ 🚀

ภารกิจ:

1. เพิ่มเอกสารของตนเองอย่างน้อย 3 ประโยค
2. สร้าง chunks และ embeddings ใหม่
3. เขียน `retrieve(question, k=3)`
4. สร้าง `chat_history` ที่เริ่มด้วย system message
5. เขียน `chat(message)` เพื่อค้น Context ก่อนถาม Qwen
6. ทดสอบถามต่อเนื่องอย่างน้อย 2 คำถาม

In [ ]:
# TODO: เพิ่มเอกสาร แล้วสร้าง clean_docs, chunks และ embeddings ใหม่
project_documents = []

## 12.1) เขียนฟังก์ชันค้นหา

`retrieve(question, k=3)` ต้องคืน list ของผลลัพธ์ โดยแต่ละรายการมี `text` และ `score`

In [ ]:
# TODO: เขียนฟังก์ชันค้นหา chunk ที่เกี่ยวข้อง
def retrieve(question, k=3):
    pass

## 12.2) เพิ่มความจำ

สร้าง `chat_history` เป็น list และใส่ system message หนึ่งรายการ เพื่อกำหนดให้ Bot ตอบจาก Context เท่านั้น

In [ ]:
# TODO: สร้าง chat_history
chat_history = []

## 12.3) เขียนฟังก์ชัน `chat()`

ลำดับการทำงาน:

1. เรียก `retrieve(message)`
2. รวมผลการค้นหาเป็น Context
3. สร้าง user prompt ที่มี Context และ Question
4. ส่ง messages ให้ `client.chat.completions.create()`
5. เก็บข้อความผู้ใช้และคำตอบไว้ใน `chat_history`
6. คืนค่า `answer`

In [ ]:
# TODO: เขียน RAG chat ที่ค้นเอกสารและจำบทสนทนา
def chat(message):
    pass

## 12.4) ทดสอบ Chatbot

ถามอย่างน้อย 2 คำถาม โดยคำถามที่สองอ้างอิงเรื่องจากคำถามแรก

In [ ]:
# TODO: เรียก chat() สองครั้งและแสดงคำตอบ
# print("Bot:", chat("คำถามแรก"))
# print("Bot:", chat("คำถามต่อเนื่อง"))

## Challenge: ล้างประวัติ

เขียน `clear_chat()` เพื่อลบประวัติเดิมและใส่ system message กลับเข้าไป

In [ ]:
# TODO: เขียน clear_chat()
def clear_chat():
    pass

## งานที่ต้องส่ง

- [ ] เอกสารอย่างน้อย 6 ประโยค
- [ ] ฟังก์ชัน `embed()` ทำงาน
- [ ] แสดงผล Top 3 พร้อมคะแนน
- [ ] Qwen ตอบจาก Context
- [ ] ฟังก์ชัน `chat()` ถามต่อเนื่องได้
- [ ] ทดลองคำถามที่ไม่มีคำตอบในเอกสาร

## สะท้อนการเรียนรู้

1. Embedding แตกต่างจากข้อความธรรมดาอย่างไร?
2. Cosine similarity ช่วยใน RAG อย่างไร?
3. หาก chunk ใหญ่หรือเล็กเกินไปจะเกิดอะไรขึ้น?
4. RAG ช่วยลดการตอบจากข้อมูลที่โมเดลคิดขึ้นเองได้อย่างไร?